# FactorFolio: Yahoo Finance Backup

**AC420 project, Jasper Lin.** Yahoo Finance often refuses requests for analyst forecasts (`eps_trend`), company info (`info`) and financial statements (`balance_sheet`, `financials`) from Streamlit Cloud's shared servers, while Colab is not blocked (the Lecture 3 and 15 to 17 notebooks ran here). This notebook saves a backup of what the app needs:
* `eps_snapshot.csv`: analyst EPS forecasts, sector, industry, Yahoo beta and forward P/E (DCF and peer comparables, Step 6)
* `financials_snapshot.csv`: the latest balance sheet and income statement items for the Altman Z-score (Step 7)

The app uses live Yahoo data when it can and falls back to these files, with their date shown, when Yahoo refuses.

**How to use:** Runtime, then Run all (about 4 minutes). When the last cell downloads the two files, upload them to the repo's `data/` folder on GitHub (Add file, then Upload files, then Commit changes) and reboot the app. Rerun it every week or so, and the day before the presentation.

Only free public Yahoo Finance data goes into this file (no WRDS data). Written with AI assistance (Claude); see my AI usage log.

### Step 1: Get the S&P 100 list from the app's repo

In [ ]:
!pip install -q --upgrade yfinance
!git clone -q https://github.com/jasperl12/AC420-project-FactorFolio-portfolio-risk-app.git factorfolio

import sys
import time
import pandas as pd
import yfinance as yf

sys.path.append('factorfolio')
from core import yahoo      # the app's own code for reading Yahoo's statement rows

sp100 = pd.read_csv('factorfolio/data/sp100_tickers.csv')
extra_tickers = ['MU']    # add any stock in your portfolio that isn't in the S&P 100
all_tickers = list(sp100['ticker']) + [t for t in extra_tickers if t not in list(sp100['ticker'])]
print(len(all_tickers), 'tickers')

### Step 2: Pull analyst EPS forecasts, company info and financial statements
The same loop as the Lecture 16 & 17 comparables notebook: `try/except` so one bad ticker can't stop the loop, `info.get(...)` instead of `info[...]`, and `eps.at['0y', 'current']` and `eps.at['+1y', 'current']` from the Lecture 15 & 16 notebook. Financial statements use the Lecture 3 pattern (`ticker.financials`, rows picked by label) through the app's `yahoo.statement_items`, which reads the latest fiscal year and gives NaN when Yahoo's row label is missing. A short pause between tickers keeps Yahoo from blocking the requests.

In [ ]:
data_list = []
fin_list = []
for symbol in all_tickers:
    stock = yf.Ticker(symbol)
    try:
        eps = stock.eps_trend
        info = stock.info
        data_list.append({'ticker': symbol,
                          'name': info.get('longName', symbol),
                          'sector': info.get('sector'),
                          'industry': info.get('industry'),
                          'eps_0y': eps.at['0y', 'current'],
                          'eps_1y': eps.at['+1y', 'current'],
                          'yahoo_beta': info.get('beta'),
                          'forward_pe': info.get('forwardPE')})
    except Exception as e:
        print(symbol, 'EPS or info failed:', e)
    try:
        fin_list.append({'ticker': symbol, **yahoo.statement_items(stock.balance_sheet, stock.financials)})
    except Exception as e:
        print(symbol, 'statements failed:', e)
    time.sleep(0.5)

today = pd.Timestamp.today().strftime('%Y-%m-%d')
snapshot = pd.DataFrame(data_list)
snapshot['as_of'] = today
financials = pd.DataFrame(fin_list)
prices = yf.download(all_tickers, period='5d', progress=False)['Close'].ffill().iloc[-1]   # for market value
financials['price'] = financials['ticker'].map(prices)
financials['as_of'] = today
print(len(snapshot), 'EPS rows and', len(financials), 'statement rows saved, of', len(all_tickers), 'tickers')

### Step 3: Look at the data before saving it
Prof. Wysocki's rule: inspect first. Check how many forecasts are missing, whether any EPS is negative (the DCF leaves those stocks out), and whether next year's EPS is far from this year's (the Lecture 16 Amazon lesson). For the statements, banks and insurers have no current assets or liabilities (Altman Z leaves them out anyway); other missing rows mean Yahoo used a different label.

In [ ]:
print(snapshot.isna().sum())
print(snapshot[['eps_0y', 'eps_1y', 'yahoo_beta', 'forward_pe']].describe().round(2))

snapshot['g'] = snapshot['eps_1y'] / snapshot['eps_0y'] - 1
print('Negative EPS:', snapshot.loc[(snapshot['eps_0y'] <= 0) | (snapshot['eps_1y'] <= 0), 'ticker'].tolist())
print('EPS falling next year:', snapshot.loc[snapshot['g'] < 0, 'ticker'].tolist())
print('Growth above 25%:', snapshot.loc[snapshot['g'] > 0.25, 'ticker'].tolist())
snapshot = snapshot.drop(columns='g')
print(financials.isna().sum())
snapshot.head(10)

### Step 4: Save and download the two files

In [ ]:
assert len(snapshot) >= 80 and len(financials) >= 80, 'Too many tickers failed: Yahoo may be blocking Colab too. Wait a few minutes and run again.'
snapshot.to_csv('eps_snapshot.csv', index=False)
financials.to_csv('financials_snapshot.csv', index=False)

from google.colab import files
files.download('eps_snapshot.csv')
files.download('financials_snapshot.csv')